In [ ]:
import os
import scanpy as sc
import numpy as np

import scvi
from scvi.external import RNAStereoscope, SpatialStereoscope

celltype_key = "celltype_final"

for dataset_number in range(1, 33):
    sc_file = f"/home/maweicheng/ST_data/SimualtedSpatalData/dataset{dataset_number}/scRNA.h5ad"
    st_file = f"/home/maweicheng/ST_data/SimualtedSpatalData/dataset{dataset_number}/Spatial.h5ad"
    output_dir = f"/home/maweicheng/ST_data/SimualtedSpatalData/evaluate/Data{dataset_number}"
    os.makedirs(output_dir, exist_ok=True)

    print(f"=== Data {dataset_number} ===")
    print(">>> Loading data")
    sc_adata = sc.read_h5ad(sc_file)
    st_adata = sc.read_h5ad(st_file)

    # ----------------- 1. 单细胞预处理（Stereoscope 版本） -----------------
    print(">>> Preprocessing scRNA")

    # 过滤掉低表达基因
    sc.pp.filter_genes(sc_adata, min_counts=10)

    # 去掉线粒体基因（MT- 开头），保持和你之前脚本一致
    non_mito_genes = [g for g in sc_adata.var_names if not g.startswith("MT-")]
    sc_adata = sc_adata[:, non_mito_genes]

    # 原始 counts 存在 layer，给 Stereoscope 用
    sc_adata.layers["counts"] = sc_adata.X.copy()

    # 归一化 + log1p（主要给可视化 / 其它分析用）
    sc.pp.normalize_total(sc_adata, target_sum=1e5)
    sc.pp.log1p(sc_adata)
    sc_adata.raw = sc_adata

    # HVG（在 counts 上选 7000 个）
    sc.pp.highly_variable_genes(
        sc_adata,
        n_top_genes=7000,
        subset=True,
        layer="counts",
        flavor="seurat_v3",
        span=1,
    )

    # ----------------- 2. 空间数据预处理 -----------------
    print(">>> Preprocessing spatial")

    # 原始 counts 存在 layer，给 Stereoscope 用
    st_adata.layers["counts"] = st_adata.X.copy()

    # 归一化 + log1p
    sc.pp.normalize_total(st_adata, target_sum=1e5)
    sc.pp.log1p(st_adata)
    st_adata.raw = st_adata

    # 基因交集（保证两个模型看到的基因一致）
    intersect = np.intersect1d(sc_adata.var_names, st_adata.var_names)
    sc_adata = sc_adata[:, intersect].copy()
    st_adata = st_adata[:, intersect].copy()
    print(f">>> Shared genes: {len(intersect)}")

    # ----------------- 3. 训练 RNAStereoscope（sc 侧） -----------------
    print(">>> Training RNAStereoscope")

    RNAStereoscope.setup_anndata(
        sc_adata,
        layer="counts",
        labels_key=celltype_key,
    )

    stereo_sc_model = RNAStereoscope(sc_adata)
    stereo_sc_model.train(max_epochs=100)

    # ----------------- 4. 训练 SpatialStereoscope（ST 侧） -----------------
    print(">>> Training SpatialStereoscope")

    SpatialStereoscope.setup_anndata(
        st_adata,
        layer="counts",
    )

    spatial_model = SpatialStereoscope.from_rna_model(st_adata, stereo_sc_model)
    spatial_model.train(max_epochs=10000)

    # ----------------- 5. 导出 proportion -----------------
    print(">>> Saving results")
    proportions = spatial_model.get_proportions()
    proportions.to_csv(os.path.join(output_dir, "Stereoscope.csv"))

    print(f">>> Data {dataset_number} done.\n")


In [ ]:
import os
import scanpy as sc
import numpy as np

import scvi
from scvi.external import RNAStereoscope, SpatialStereoscope


celltype_key = "celltype"
sc_file = f"/home/maweicheng/ST_data/STARmap/starmap_sc_rna.h5ad"
st_file = f"/home/maweicheng/ST_data/STARmap/STARmap_SP.h5ad"
output_dir = f"/home/maweicheng/ST_data/STARmap"
os.makedirs(output_dir, exist_ok=True)

print(">>> Loading data")
sc_adata = sc.read_h5ad(sc_file)
st_adata = sc.read_h5ad(st_file)

# ----------------- 1. 单细胞预处理（Stereoscope 版本） -----------------
print(">>> Preprocessing scRNA")

# 过滤掉低表达基因
sc.pp.filter_genes(sc_adata, min_counts=10)

# 去掉线粒体基因（MT- 开头），保持和你之前脚本一致
non_mito_genes = [g for g in sc_adata.var_names if not g.startswith("MT-")]
sc_adata = sc_adata[:, non_mito_genes]

# 原始 counts 存在 layer，给 Stereoscope 用
sc_adata.layers["counts"] = sc_adata.X.copy()

# 归一化 + log1p（主要给可视化 / 其它分析用）
sc.pp.normalize_total(sc_adata, target_sum=1e5)
sc.pp.log1p(sc_adata)
sc_adata.raw = sc_adata

# HVG（在 counts 上选 7000 个）
sc.pp.highly_variable_genes(
    sc_adata,
    n_top_genes=7000,
    subset=True,
    layer="counts",
    flavor="seurat_v3",
    span=1,
)

# ----------------- 2. 空间数据预处理 -----------------
print(">>> Preprocessing spatial")

# 原始 counts 存在 layer，给 Stereoscope 用
st_adata.layers["counts"] = st_adata.X.copy()

# 归一化 + log1p
sc.pp.normalize_total(st_adata, target_sum=1e5)
sc.pp.log1p(st_adata)
st_adata.raw = st_adata

# 基因交集（保证两个模型看到的基因一致）
intersect = np.intersect1d(sc_adata.var_names, st_adata.var_names)
sc_adata = sc_adata[:, intersect].copy()
st_adata = st_adata[:, intersect].copy()
print(f">>> Shared genes: {len(intersect)}")

# ----------------- 3. 训练 RNAStereoscope（sc 侧） -----------------
print(">>> Training RNAStereoscope")

RNAStereoscope.setup_anndata(
    sc_adata,
    layer="counts",
    labels_key=celltype_key,
)

stereo_sc_model = RNAStereoscope(sc_adata)
stereo_sc_model.train(max_epochs=100)

# ----------------- 4. 训练 SpatialStereoscope（ST 侧） -----------------
print(">>> Training SpatialStereoscope")

SpatialStereoscope.setup_anndata(
    st_adata,
    layer="counts",
)

spatial_model = SpatialStereoscope.from_rna_model(st_adata, stereo_sc_model)
spatial_model.train(max_epochs=10000)

# ----------------- 5. 导出 proportion -----------------
print(">>> Saving results")
proportions = spatial_model.get_proportions()
proportions.to_csv(os.path.join(output_dir, "Stereoscope.csv"))

print(f">>> Data {dataset_number} done.\n")
